# 🚀 Herramienta Definitiva: Poligonización Automática de Lotes (SAM + Sentinel-2)
**Optimizada para Google Colab (GPU T4)**

Esta notebook fusiona las mejores técnicas analizadas en el estudio de mercado y versiones anteriores:
1. **Sentinel-2 Surface Reflectance (S2_SR)**: Para cálculos de NDVI más puros y precisos.
2. **Bounding Box Dinámico**: El contexto de la imagen enviada a SAM se ajusta al área esperada del lote.
3. **Control de Fugas Avanzado**: Si SAM sobre-segmenta (se escapa del lote), inyecta automáticamente **puntos negativos** en los bordes esperados.
4. **Suavizado de Contornos (Vectorización Óptima)**: Usa `cv2.approxPolyDP` para evitar bordes pixelados y crear geometrías limpias listas para QGIS/GeoJSON.
5. **Guardado Incremental (Checkpoints)**: Tolerancia a fallos, guarda el progreso automáticamente.


In [ ]:
# 1. INSTALACIONES Y PREPARACIÓN DEL ENTORNO
# Ejecutar esta celda y luego reiniciar el entorno si Colab lo pide.
!pip install -q segment-anything geemap geopandas rasterio shapely folium tqdm openpyxl scipy opencv-python-headless
print("Librerías instaladas correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 37.5 MB/s eta 0:00:00
Librerías instaladas correctamente.


In [ ]:
# 2. IMPORTACIÓN DE LIBRERÍAS
import ee, geemap
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from segment_anything import sam_model_registry, SamPredictor
import geopandas as gpd, folium
from tqdm.notebook import tqdm
import zipfile, os, re, json, time, cv2, traceback, warnings, math
from scipy.ndimage import binary_erosion, binary_dilation
from shapely.geometry import Polygon, mapping, shape
import matplotlib.cm as cm

warnings.filterwarnings('ignore')
print("Librerías importadas.")

Librerías importadas.


In [ ]:
# 3. CONFIGURACIÓN GENERAL DEL PIPELINE
CONFIG = {
    'GEE_PROJECT': 'applied-oxygen-459415-e2',   # Reemplazar con tu proyecto de Earth Engine
    'BUFFER_M': 2500,                            # Tamaño del parche descargado de GEE
    'MAX_NUBES': 20,                             # % máximo de nubes tolerado en la ventana
    'DIAS_VENTANA': 60,                          # Ventana temporal para buscar imagen pre-evento limpia
    'FECHA_FALLBACK_INICIO': '2025-11-01',       # Fecha fallback si no hay fecha en el CSV
    'FECHA_FALLBACK_FIN': '2026-03-31',
    'MIN_AREA_HA': 2,                            # Área mínima admisible del polígono
    'MAX_AREA_HA': 1500,                         # Área máxima admisible (previene super-segmentaciones)
    'TOLERANCIA_FUGA': 1.5,                      # Si SAM detecta > 1.5x el daño, intenta corregir
    'MARGEN_PX_MINIMO': 20,                      # Box mínimo dinámico
    'MARGEN_PX_MAXIMO': 100,                     # Box máximo dinámico
    'GUARDAR_CADA': 5,                           # Checkpoints cada N lotes
    'FACTOR_SUAVIZADO': 0.005,                   # Nivel de simplificación del contorno
}
print("Configuración cargada.")

Configuración cargada.


In [ ]:
# 4. CARGA Y LIMPIEZA ROBUSTA DE DATOS
# Sube tu CSV de siniestros
from google.colab import files
print("Por favor, sube el archivo CSV de lotes/siniestros:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df_raw = pd.read_csv(filename)

# Identificación automática de columnas comunes
def encontrar_columna(patrones, df):
    for p in patrones:
        for col in df.columns:
            if re.search(p, str(col), re.IGNORECASE):
                return col
    return None

rename_dict = {}
for var, patrones in [
    ('id', ['taype', 'id']),
    ('lat', ['lat_dec', 'latitude', 'lat', 'cg_latitud']),
    ('lon', ['lon_dec', 'longitude', 'lon', 'cg_longitu']),
    ('fecha', ['fecha_de_s', 'fecha', 'date']),
    ('dano_ha', ['has__daña', 'dano_ha', 'dano_estimado', 'area_ha', 'sup_afectada_ha']),
    ('cultivo', ['cultivo', 'crop']),
    ('localidad', ['localidad', 'location'])
]:
    col = encontrar_columna(patrones, df_raw)
    if col: rename_dict[col] = var

df = df_raw.rename(columns=rename_dict)

def parse_decimal(val):
    if pd.isna(val): return np.nan
    if isinstance(val, (int, float)): return float(val)
    s = str(val).strip().replace(',', '.')
    if s.count('.') > 1: s = ''.join(s.split('.')[:-1]) + '.' + s.split('.')[-1]
    try: return float(s)
    except: return np.nan

for col in ['dano_ha', 'lat', 'lon']:
    if col in df.columns: df[col] = df[col].apply(parse_decimal)

if 'fecha' in df.columns:
    df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce', dayfirst=True)
else:
    df['fecha'] = pd.NaT

df = df.dropna(subset=['lat', 'lon'])
if 'id' not in df.columns: df['id'] = range(1, len(df)+1)

# Reducción espacial (evita procesar el mismo punto dos veces)
df['lat_r'] = df['lat'].round(4)
df['lon_r'] = df['lon'].round(4)
df_geo = df.sort_values('id').drop_duplicates(subset=['lat_r','lon_r']).reset_index(drop=True)

# Lógica de Fechas Seguras (Fallback si NaT)
def set_fecha(r, is_inicio=True):
    if pd.isna(r['fecha']):
        return pd.to_datetime(CONFIG['FECHA_FALLBACK_INICIO'] if is_inicio else CONFIG['FECHA_FALLBACK_FIN'])
    else:
        return r['fecha'] - pd.Timedelta(days=CONFIG['DIAS_VENTANA']) if is_inicio else r['fecha'] - pd.Timedelta(days=1)

df_geo['fecha_inicio'] = df_geo.apply(lambda r: set_fecha(r, True), axis=1)
df_geo['fecha_fin'] = df_geo.apply(lambda r: set_fecha(r, False), axis=1)

print(f"CSV cargado. Lotes espaciales únicos a procesar: {len(df_geo)}")
display(df_geo[['id', 'localidad', 'cultivo', 'dano_ha', 'lat', 'lon', 'fecha_inicio']].head())

Por favor, sube el archivo CSV de lotes/siniestros:


Saving AgroIA_Dataset_Maestro_Siniestros.csv to AgroIA_Dataset_Maestro_Siniestros.csv
CSV cargado. Lotes espaciales únicos a procesar: 313


,id,localidad,cultivo,dano_ha,lat,lon,fecha_inicio
0,TAYPE,IDIAZABAL,Trigo,40.0,-32.944817,-63.164883,2018-07-20
1,TAYPE,INTENDENTE ALVEAR,Maíz,105.0,-35.123750,-63.387600,2018-11-06
2,TAYPE,INTENDENTE ALVEAR,Soja,85.0,-35.202800,-63.450550,2018-10-15
3,TAYPE,PORVENIR,Soja,50.0,-35.039700,-62.230833,2018-10-12
4,TAYPE,DRABBLE,Trigo,205.0,-34.918783,-62.820667,2018-10-11


In [ ]:
# 5. VALIDACIÓN EN MAPA INTERACTIVO (Folium)
centro = [df_geo['lat'].mean(), df_geo['lon'].mean()]
mapa = folium.Map(location=centro, zoom_start=6, tiles='CartoDB positron')

for _, row in df_geo.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=6, color='#2980b9', fill=True, fill_opacity=0.8,
        popup=f"<b>ID:</b> {row['id']}<br><b>Loc:</b> {row.get('localidad','')}<br><b>Daño:</b> {row.get('dano_ha', 'N/A')} ha"
    ).add_to(mapa)

mapa.save('puntos_a_procesar.html')
print("Mapa generado. Revisa que los puntos tengan sentido geográfico:")
display(mapa)

Mapa generado. Revisa que los puntos tengan sentido geográfico:


In [ ]:
# 6. INICIALIZACIÓN DE SERVICIOS (Earth Engine + Segment Anything)
# GEE
try:
    ee.Initialize(project=CONFIG['GEE_PROJECT'])
    print("GEE conectado.")
except:
    ee.Authenticate()
    ee.Initialize(project=CONFIG['GEE_PROJECT'])

# SAM
print("Descargando modelo SAM (si no existe)...")
!wget -q -nc https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

print("Cargando SAM en GPU...")
sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b_01ec64.pth")
sam.to("cuda")
predictor = SamPredictor(sam)
print("SAM inicializado con éxito en T4!")

Descargando modelo SAM (si no existe)...
Cargando SAM en GPU...
SAM inicializado con éxito en T4!


In [ ]:
# 7. FUNCIONES NÚCLEO (Extracción GEE, Bounding Box y SAM Avanzado)
def extraer_imagen_sr(lat, lon, fecha_inicio, fecha_fin):
    """Descarga de Sentinel-2 Surface Reflectance (mejor para NDVI)."""
    centro = ee.Geometry.Point([lon, lat])
    bbox = centro.buffer(CONFIG['BUFFER_M']).bounds()
    coleccion = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterBounds(bbox)
                 .filterDate(fecha_inicio.strftime('%Y-%m-%d'), fecha_fin.strftime('%Y-%m-%d'))
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CONFIG['MAX_NUBES']))
                 .sort('CLOUDY_PIXEL_PERCENTAGE'))

    if coleccion.size().getInfo() == 0: return None, None
    img = coleccion.first()
    datos = img.select(['B4','B8']).sampleRectangle(region=bbox, defaultValue=0)
    rojo = np.array(datos.get('B4').getInfo(), dtype=np.float32)
    nir  = np.array(datos.get('B8').getInfo(), dtype=np.float32)
    ndvi = (nir - rojo) / (nir + rojo + 1e-6)
    return ndvi, bbox

def ndvi_a_rgb(ndvi):
    ndvi_clip = np.clip(ndvi, -0.2, 0.8)
    ndvi_norm = ((ndvi_clip + 0.2) * 255).astype(np.uint8)
    cmap = cm.get_cmap('RdYlGn')
    return (cmap(ndvi_norm / 255.0)[:, :, :3] * 255).astype(np.uint8)

def segmentar_inteligente(row):
    """Aplica SAM con BBox dinámico, inyección de puntos negativos y suavizado."""
    ndvi, bbox = extraer_imagen_sr(row['lat'], row['lon'], row['fecha_inicio'], row['fecha_fin'])
    if ndvi is None: return None, 0, 0, 'SIN_IMAGEN'

    h, w = ndvi.shape
    area_ref = row.get('dano_ha', 50)
    if pd.isna(area_ref) or area_ref <= 0: area_ref = 50

    # Bounding Box Dinámico y Clamped
    lado_m = np.sqrt(area_ref * 10000)
    margen = int((lado_m / 2) / 10)
    margen = max(CONFIG['MARGEN_PX_MINIMO'], min(margen, CONFIG['MARGEN_PX_MAXIMO']))
    cx, cy = w//2, h//2
    x_min, x_max = max(0, cx-margen), min(w-1, cx+margen)
    y_min, y_max = max(0, cy-margen), min(h-1, cy+margen)
    box = np.array([x_min, y_min, x_max, y_max])

    predictor.set_image(ndvi_a_rgb(ndvi))

    # Intento 1: Box normal + Prompt central
    masks, scores, _ = predictor.predict(
        point_coords=np.array([[cx, cy]]), point_labels=np.array([1]),
        box=box[None, :], multimask_output=True
    )
    mask = masks[np.argmax(scores)]
    area_sam = (mask.sum() * 100) / 10000

    # Control de fuga (Leakage): Puntos negativos en las esquinas del BBOX
    if area_sam > area_ref * CONFIG['TOLERANCIA_FUGA']:
        pts_neg = np.array([[x_min, y_min], [x_max, y_min], [x_max, y_max], [x_min, y_max]])
        todas_coords = np.vstack([[cx, cy], pts_neg])
        todas_labels = np.array([1, 0, 0, 0, 0])
        masks2, scores2, _ = predictor.predict(
            point_coords=todas_coords, point_labels=todas_labels,
            box=box[None, :], multimask_output=False
        )
        mask = masks2[0]
        area_sam = (mask.sum() * 100) / 10000
        if area_sam > area_ref * CONFIG['TOLERANCIA_FUGA']:
            return None, area_sam, scores2[0], 'FUGA_CRÍTICA'

    # Vectorización Suavizada
    mask = binary_erosion(mask, iterations=1)
    mask = binary_dilation(mask, iterations=2)
    contornos, _ = cv2.findContours((mask*255).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contornos: return None, area_sam, np.max(scores), 'SIN_CONTORNO'

    contorno = max(contornos, key=cv2.contourArea)
    epsilon = CONFIG['FACTOR_SUAVIZADO'] * cv2.arcLength(contorno, True)
    contorno_simple = cv2.approxPolyDP(contorno, epsilon, True)
    puntos = contorno_simple.squeeze()
    if puntos.ndim == 1: puntos = puntos.reshape(1, -1)
    if len(puntos) < 3: return None, area_sam, np.max(scores), 'POLIGONO_INVALIDO'

    # Georreferenciación
    coords_b = bbox.bounds().getInfo()['coordinates'][0]
    lon_min, lon_max = min(c[0] for c in coords_b), max(c[0] for c in coords_b)
    lat_min, lat_max = min(c[1] for c in coords_b), max(c[1] for c in coords_b)

    vertices_geo = [
        (lon_min + (p[0]/w)*(lon_max-lon_min), lat_max - (p[1]/h)*(lat_max-lat_min))
        for p in puntos
    ]
    poligono = Polygon(vertices_geo).buffer(0)

    if not poligono.is_valid: return None, area_sam, np.max(scores), 'GEOM_INVALIDA'
    area_ha = poligono.area * (111320 ** 2) * math.cos(math.radians(row['lat'])) / 10000

    if area_ha < CONFIG['MIN_AREA_HA']: return None, area_ha, np.max(scores), 'AREA_MUY_CHICA'
    if area_ha > CONFIG['MAX_AREA_HA']: return None, area_ha, np.max(scores), 'AREA_MUY_GRANDE'

    return poligono, area_ha, np.max(scores), 'OK'


In [ ]:
OUTPUT_FILE = 'poligonos_definitivos.geojson'
LOG_FILE = 'pipeline_log.csv'

# Checkpointing
procesados = set()
features = []
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE) as f:
        features = json.load(f).get('features', [])
    procesados = {ft['properties']['id'] for ft in features}

log_rows = []
if os.path.exists(LOG_FILE):
    log_rows = pd.read_csv(LOG_FILE).to_dict('records')
    procesados |= {r['id'] for r in log_rows}

pendientes = df_geo[~df_geo['id'].isin(procesados)].reset_index(drop=True)

print(f"Checkpoint: {len(procesados)} listos. Restantes: {len(pendientes)}.")
print("-" * 75)

if len(pendientes) > 0:
    for idx, row in pendientes.iterrows():
        t0 = time.time()
        poly, area, score, status = segmentar_inteligente(row)
        t_elap = time.time() - t0

        area_ref = row.get('dano_ha', 50)
        if pd.isna(area_ref) or area_ref <= 0: area_ref = 50
        error_pct = abs(area - area_ref) / area_ref * 100 if area else 0

        log_rows.append(
            {
                'id': row['id'],
                'estado': status,
                'area_calc': area,
                'sam_score': score,
                'error_pct': error_pct,
                'segundos': round(t_elap, 1),
            }
        )

        if status == 'OK' and poly is not None:
            features.append({
                "type": "Feature",
                "properties": {
                    "id": str(row['id']), # Changed int(row['id']) to str(row['id'])
                    "localidad": row.get('localidad', ''),
                    "cultivo": row.get('cultivo', ''),
                    "area_ha": round(area, 2),
                    "error_pct": round(error_pct, 1),
                    "sam_score": round(float(score), 3)
                },
                "geometry": mapping(poly)
            })
            print(f"[{idx+1}/{len(pendientes)}] ID: {row['id']} | {area:.1f} ha | Score: {score:.2f} | Error: {error_pct:.1f}% | {t_elap:.1f}s")
        else:
            print(f"[{idx+1}/{len(pendientes)}] ID: {row['id']} | Falló: {status} (Área: {area:.1f} ha)")

        # Guardado incremental
        if (idx + 1) % CONFIG['GUARDAR_CADA'] == 0 or idx == len(pendientes) - 1:
            with open(OUTPUT_FILE, 'w') as f:
                json.dump({"type": "FeatureCollection", "features": features}, f, indent=2)
            pd.DataFrame(log_rows).to_csv(LOG_FILE, index=False)

print("-" * 75)
print(f"Pipeline Finalizado. Se guardaron {len(features)} polígonos válidos en {OUTPUT_FILE}.")

Checkpoint: 0 listos. Restantes: 313.
---------------------------------------------------------------------------
[1/313] ID: TAYPE | Falló: SIN_IMAGEN (Área: 0.0 ha)
[2/313] ID: TAYPE | 72.6 ha | Score: 0.94 | Error: 30.9% | 3.4s
[3/313] ID: TAYPE | 59.9 ha | Score: 0.90 | Error: 29.5% | 3.5s
[4/313] ID: TAYPE | 34.7 ha | Score: 0.88 | Error: 30.7% | 3.6s
[5/313] ID: TAYPE | 62.6 ha | Score: 0.86 | Error: 69.5% | 3.6s
[6/313] ID: TAYPE | 66.1 ha | Score: 0.95 | Error: 51.0% | 3.4s
[7/313] ID: TAYPE | 48.7 ha | Score: 0.85 | Error: 41.3% | 3.2s
[8/313] ID: TAYPE | 13.5 ha | Score: 0.92 | Error: 38.7% | 3.4s
[9/313] ID: TAYPE | 7.2 ha | Score: 0.94 | Error: 34.4% | 3.3s
[10/313] ID: TAYPE | 7.8 ha | Score: 0.91 | Error: 34.7% | 3.1s
[11/313] ID: TAYPE | 164.9 ha | Score: 0.81 | Error: 79.9% | 3.1s
[12/313] ID: TAYPE | 41.4 ha | Score: 0.93 | Error: 51.3% | 3.0s
[13/313] ID: TAYPE | 143.7 ha | Score: 0.95 | Error: 42.3% | 3.6s
[14/313] ID: TAYPE | 89.1 ha | Score: 0.90 | Error: 41.0% | 4

In [ ]:
# 9. DESCARGA DE RESULTADOS
from google.colab import files
import os

print("Descargando archivos generados...")
if os.path.exists(OUTPUT_FILE):
    files.download(OUTPUT_FILE)
if os.path.exists(LOG_FILE):
    files.download(LOG_FILE)
if os.path.exists('puntos_a_procesar.html'):
    files.download('puntos_a_procesar.html')

Descargando archivos generados...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>